# TRELLIS.2 Model Completion Pipeline

Break down the completion pipeline into separate steps for easier testing and debugging.

In [10]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True"
)  # Can save GPU memory
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

import trimesh
import torch
import numpy as np
from PIL import Image
from trellis2.pipelines.trellis2_model_completion import Trellis2ModelCompletionPipeline
from trellis2.utils.mesh_utils import get_scene_geometry
from trellis2.utils.vox_utils import mesh_to_voxel_volume
import o_voxel
from trellis2.modules.sparse import SparseTensor

## Load Pipeline

In [4]:
pipeline = Trellis2ModelCompletionPipeline.from_pretrained(
    "/home/cyvv/share/.cache/huggingface/hub/models--microsoft--TRELLIS.2-4B/snapshots/af44b45f2e35a493886929c6d786e563ec68364d",
    config_file="pipeline_completion.json",
)
pipeline.cuda()
print("Pipeline loaded and moved to CUDA")

Loading weights: 100%|██████████| 754/754 [00:01<00:00, 473.07it/s, Materializing param=squeeze_module.0.dec_att.global_avg_pool.2.weight]                              


Pipeline loaded and moved to CUDA


## Load Mesh and Image

In [7]:
# Load base mesh
MESH_PATH = "/home/cyvv/share/Research/trellis-modeller/r2dhorse_noperm.glb"
REGION_PATH = "/home/cyvv/share/Research/trellis-modeller/r2dhorse_inpaint_region_noperm.glb"
IMAGE_PATH = "/home/cyvv/share/Research/trellis-modeller/r2dhorse_render_inpainted_scarf.png"
base = trimesh.load(MESH_PATH)

# Load inpaint region
inpaint_region = trimesh.load(REGION_PATH)

# Load conditioning image
image = Image.open(IMAGE_PATH)
print(f"Image loaded: {image.size}")

Image loaded: (868, 868)


## Override Tex Slat sampling

In [8]:
# Store original tex slat
_original_sample_tex_slat = pipeline.sample_tex_slat

In [11]:
# Override function
def o_sample_tex_slat(
    self,
    ss_coords: torch.Tensor,
    target: SparseTensor,
    region: SparseTensor,
    cond: dict,
    flow_model,
    shape_slat: SparseTensor,
    sampler_params: dict = {},
) -> SparseTensor:
    """
    Sample structured latent with the given conditioning.

    Args:
        cond (dict): The conditioning information.
        shape_slat (SparseTensor): The structured latent for shape
        sampler_params (dict): Additional parameters for the sampler.
    """
    # Sample structured latent
    std = torch.tensor(self.shape_slat_normalization["std"])[None].to(
        shape_slat.device
    )
    mean = torch.tensor(self.shape_slat_normalization["mean"])[None].to(
        shape_slat.device
    )
    shape_slat = (shape_slat - mean) / std
    std = torch.tensor(self.tex_slat_normalization["std"])[None].to(
        shape_slat.device
    )
    mean = torch.tensor(self.tex_slat_normalization["mean"])[None].to(
        shape_slat.device
    )
    target = (target - mean) / std

    noise = self.tex_slat_sampler.prepare_inpainting(
        target, region, template=shape_slat.coords
    )
    # noise = SparseTensor(
    #     feats=torch.randn(
    #         (ss_coords.shape[0], target.feats.shape[1]), device=target.device
    #     ),
    #     coords=ss_coords,
    # )

    sampler_params = {**self.tex_slat_sampler_params, **sampler_params}
    if self.low_vram:
        flow_model.to(self.device)
    slat = self.tex_slat_sampler.sample(
        flow_model,
        noise,
        concat_cond=shape_slat,
        **cond,
        **sampler_params,
        verbose=True,
        tqdm_desc="Sampling texture SLat",
    ).samples
    if self.low_vram:
        flow_model.cpu()

    slat = slat * std + mean

    return slat

pipeline.sample_tex_slat = o_sample_tex_slat


In [ ]:
# Revert sampling
pipeline.sample_tex_slat = _original_sample_tex_slat

## Run Completion Pipeline

In [12]:
%%time
OUT_PATH = "sample_inpaint_1.glb"
mesh = pipeline.run(
    base,
    image,
    inpaint_region
)[0]
glb = o_voxel.postprocess.to_glb(
    vertices=mesh.vertices,
    faces=mesh.faces,
    attr_volume=mesh.attrs,
    coords=mesh.coords,
    attr_layout=mesh.layout,
    voxel_size=mesh.voxel_size,
    aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
    decimation_target=1000000,
    texture_size=4096,
    remesh=True,
    remesh_band=1,
    remesh_project=0,
    verbose=True,
)
glb.export(OUT_PATH, extension_webp=True)

Intersect QEF computation took 0.947728 seconds.
Face QEF computation took 0.242449 seconds.
Boundary QEF computation took 0.049243 seconds.
Dual vertices computation took 0.208994 seconds.
Voxelize settings:
  Voxel size: tensor([0.0010, 0.0010, 0.0010])
  Grid size: tensor([1024, 1024, 1024], dtype=torch.int32)
  AABB: tensor([[-0.5000, -0.5000, -0.5000],
        [ 0.5000,  0.5000,  0.5000]])


Loading Scene:   0%|          | 0/2 [00:00<?, ?it/s]

Geometry: defaultMaterial.007
  Visual: <trimesh.visual.texture.TextureVisuals object at 0x70ec5427a830>
  Triangles: 5324
  Vertices: 3559
  Normals: 3559
  Base color texture: (2048, 2048) RGB
  Metallic roughness texture: (2048, 2048) RGB
  Emissive factor: [1. 1. 1.]
  Emissive texture: (2048, 2048) RGB
  Normal texture: (2048, 2048) RGB


Loading Scene: 100%|██████████| 2/2 [00:00<00:00,  4.28it/s]


Geometry: defaultMaterial.007_1
  Visual: <trimesh.visual.texture.TextureVisuals object at 0x70ec5427bc70>
  Triangles: 128794
  Vertices: 71555
  Normals: 71555
  Base color texture: (2048, 2048) RGB
  Metallic factor: 0.0
  Roughness factor: 0.7136363983154297
Mipmaps construction took 0.530803 seconds.
Voxelization took 1.66282 seconds.
Normalization took 0.014819 seconds.


Sampling shape SLat: 100%|██████████| 12/12 [00:08<00:00,  1.40it/s]


AttributeError: 'Tensor' object has no attribute 'shape_slat_normalization'